# 02 · ResNet-50, ResNet-50 + morphometrics, and the morphometric-only MLP

Trained with the package's own sweep, `tools/train_eval_all.py`, at its defaults: ImageNet
ResNet-50, seed 95, deterministic cuDNN kernels, checkpoint chosen on the validation animals.
A ResNet-50 retrain is not bit-identical to the shipped model: threaded patch loading makes
the random-flip order timing-dependent (see notebook 09). The MLP retrains identically.
`cnn_morph` concatenates the 2048-d image embedding with a small projection of the two
z-scored slide-level morphometrics (mean myofiber CSA, myofiber count); `mlp_morph` sees
the morphometrics only (the Table 2 ablation).

Each ResNet-50 run takes 5–40 min on the reference GPU (≈ 1.5 h for all nine runs).

**`RUN`** decides what this notebook does.
* With the original data on disk, `False` (default) reads the results of the run behind
  the manuscript and checks them; `True` re-trains into `runs/rerun`.
* On the synthetic data, `RUN` is `True`: there are no saved results to read, so the models
  are trained (into `runs/synthetic/rerun`; finished runs are skipped).

The manuscript's results folders are never overwritten, and nothing is written into the
dataset.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
RUN = SYNTHETIC                           # <- True: train into RERUN_BASE (always on synthetic data)
RERUN_BASE = P.WORK / "rerun"
BASE = RERUN_BASE if RUN else P.RESULTS   # where this notebook reads results from

In [ ]:
RES = P.roots(BASE)["cnn"]
if RUN:
    cmd = [PY, "-u", str(P.VML / "tools" / "train_eval_all.py"), "--data-root", str(P.DATA),
           "--results-root", str(RES), "--models", "cnn,cnn_morph,mlp_morph"]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(P.VML), check=True)   # resumable: finished runs are skipped
print("results:", RES)

In [ ]:
from manuscript_numbers import compute
N = compute(BASE, models=["cnn", "cnn_morph", "mlp_morph"], human=[])
pd.DataFrame(N["rat"]).T[["patch_f1_pos", "wsi_f1_pos", "patch_macro", "wsi_macro", "auc", "n_patch", "n_wsi"]].round(3)

`patch_f1_pos` / `wsi_f1_pos` are the F1 of the more injured class (VML; Very poor), the
value the manuscript reports for the binary tasks; for three-class it reports macro-F1.